# Описание подхода и пайплайн решения

**Задача**
Бинарная классификация cookie-сессий: нам нужно определить, является ли сессия ботом (target=1) или человеком (target=0). Доля ботов в train ≈ 8.1% -> имеем дисбаланс классов, некритичный, но его важно учитывать.
**Целевая метрика** — Precision@Recall=0.7.

## Пайплайн
### 1. Загрузка и объединение таблиц
Источники:
- `train.csv` / `test.csv` — метаданные cookie: `cookie_id`, `cookie_created_at`, окно `[window_start_ts, window_end_ts]`, `target` (только в train).
- `events.csv.gz` — лог событий 

**`attach_events`** — inner-join по `cookie_id` + фильтр по временному окну: оставляем только события, попавшие строго внутрь `[window_start_ts, window_end_ts]` каждого cookie. Это ключевой шаг: одна и та же сессия cookie может появляться в разных окнах, и без фильтра был бы data leakage через события вне окна.

### 2. Feature Engineering (`build_features`)
Агрегация событий **на уровне cookie**. Группировка `groupby('cookie_id')`, дальше построение ~43 признаков по 6 смысловым блокам:

**a) UA-признаки** (парсинг через `user_agents.parse`):

**b) Поведение мыши / клики:**
- `n_clicks`, `n_unique_points` — число кликов и уникальных точек.
- `avg_distance` — среднее pairwise-расстояние между точками кликов. Бот скорее всего либо кликает в одну точку (distance≈0), либо хаотично по всей странице.

**c) Временные признаки:**
- `n_events_per_min`, `std_inter_event` — плотность и разброс интервалов событий.
- `cv_inter_event` = std/mean интервалов, `regularity = 1 - cv` — боты часто кликают с равномерным интервалом, а человек неравномерно (cv→0, regularity→1).
- `entropy_minute` — нормированная энтропия распределения событий по минутам. У ботов активность либо строго равномерная, либо сконцентрирована.

**d) Навигация, просмотренные страницы, просмотренные регионы и товары:**
- `n_visited_pages`, `max_page` — глубина просмотра.
- `n_unique_locations`, `avg_interactions_per_location` — разнообразие позиций на странице.
- `n_unique_categories`, `n_unique_items`, `n_events_with_item_id`.

**e) Поисковые запросы:**
- `n_query_events`, `n_unique_query` — интенсивность и разнообразие поисковых запросов.

**f) Событийные фичи:**
- `n_unique_event_names`.
- `cnt_<event_name>` — счётчики по каждому типу события.
- `event_name_bin_<k>` — счётчики, сгруппированные по бинам индексов (idx // 10) от частотного ранжирования event_name. Это снижает размерность и борется с редкими событиями.


### 5. Подготовка для моделей
- `X`, `y` из `agg_full`.
- Пропуски можно не обрабатывать, градиентному бустингу надеревьях, которым являются выбранные модели, они не мешают, а работают даже как отдельная категория. Так же им не нужна стандартизация числовых данных.


### 6. Метрика

**precision@recall(0.7)**

## Модели
Следующие модели были выбраны по ряду причин - градиентны бустинг на деревьях это база современного classic-ml. Они эффективны, работают быстро (особенно LGBM), хорошо обрабатывают большие обьемы данных (XGBoost). Более того они отлично работают с категориальными признаками и не требуют их специальной предобработки.
1. **LightGBM** (основная модель) - baseline (позже усилим ее блендингом с XGBoost).

2. **XGBoost** (вторая модель для блендинга) (сравним с LightGBM, позже сблендим с ней).

Для подбора параметров применим Optuna.

### Кросс-валидация
Обе модели обучаются по **StratifiedKFold**:
- OOF-предсказания на всём train.
- Test-предсказания усредняются по 5 фолдам.
- Каждый фолд сохраняет свой `best_iteration_` (без этого — типичная ошибка, приводящая к переобучению на test).


### 6. Взвешенный блендинг через ранги
Дает устойчивость к разнице калибровок LGB и XGB.

Подбор весов через Optuna.

## Импортируем необходимые библиотеки

In [16]:
import numpy as np
import pandas as pd
import itertools
import time
import warnings
warnings.filterwarnings("ignore")
import optuna
from user_agents import parse
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
import lightgbm as lgb
RANDOM_SEED = 42 #для воспроизводимости
TARGET_RECALL = 0.7

## Загрузим данные и объединим таблицы

In [2]:
train  = pd.read_csv('data/train.csv',  parse_dates=['cookie_created_at','window_start_ts','window_end_ts'])
test   = pd.read_csv('data/test.csv',   parse_dates=['cookie_created_at','window_start_ts','window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))


def attach_events(meta: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame: #соединяем 
    """Приклеивает события к метаданным и оставляет только те, что внутри окна."""
    df = meta.merge(events, on='cookie_id', how='left', suffixes=('', '_ev'))
    mask = (df['event_ts'] > df['window_start_ts']) & (df['event_ts'] < df['window_end_ts'])
    return df.loc[mask].reset_index(drop=True)


train_ev = attach_events(train, events)
test_ev  = attach_events(test,  events)

print('train_ev:', train_ev.shape, '| test_ev:', test_ev.shape)
print('cookie без событий (train):', train.cookie_id.nunique() - train_ev.cookie_id.nunique())
print('cookie без событий (test): ', test.cookie_id.nunique()  - test_ev.cookie_id.nunique())

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811
train_ev: (198436, 18) | test_ev: (89690, 17)
cookie без событий (train): 0
cookie без событий (test):  0


In [119]:
#проверка дубликатов
events.shape, events.nunique

((328905, 14),
 <bound method DataFrame.nunique of                   cookie_id            event_ts  eid           event_name  \
 0       ck_5efbea1befdefe1b 2026-04-26 09:11:24  200            item_view   
 1       ck_c4ca1434f3778f1d 2026-04-20 14:04:31  100  search_results_view   
 2       ck_d274382b19488771 2026-04-13 12:02:56  200            item_view   
 3       ck_fbbed2ff14944ce9 2026-04-08 06:12:14  100  search_results_view   
 4       ck_56cc15c7c634cb9f 2026-04-12 17:16:05  100  search_results_view   
 ...                     ...                 ...  ...                  ...   
 328900  ck_92b6cdb286349983 2026-04-17 12:03:28  210          photo_swipe   
 328901  ck_0d7e121f3f699cab 2026-04-17 09:08:06  100  search_results_view   
 328902  ck_94fff69584dd3b2a 2026-04-26 21:09:15  200            item_view   
 328903  ck_141170783b107b5f 2026-04-10 07:27:07  100  search_results_view   
 328904  ck_212380e0371d8a6d 2026-04-21 19:15:57  300   contact_phone_show   
 
        plat

In [120]:
train.shape, train.nunique

((11091, 5),
 <bound method DataFrame.nunique of                  cookie_id   cookie_created_at window_start_ts window_end_ts  \
 0      ck_54a059eb7d3ea68b 2025-11-21 09:30:41      2026-04-06    2026-04-07   
 1      ck_7e4de46eeab82974 2025-09-23 10:10:24      2026-04-06    2026-04-07   
 2      ck_9320229ef6304522 2026-03-04 00:08:02      2026-04-06    2026-04-07   
 3      ck_30ccd25bc1714ed9 2026-04-05 10:52:40      2026-04-06    2026-04-07   
 4      ck_a77c5f05948cdeef 2026-01-01 03:54:35      2026-04-06    2026-04-07   
 ...                    ...                 ...             ...           ...   
 11086  ck_2256c19bb847dd0e 2026-04-18 03:59:35      2026-04-19    2026-04-20   
 11087  ck_ba2e403201cba06d 2026-03-11 01:30:00      2026-04-19    2026-04-20   
 11088  ck_2f43054a1f52b9a2 2026-03-19 03:44:55      2026-04-19    2026-04-20   
 11089  ck_354f57a870e41ddd 2026-04-17 22:44:59      2026-04-19    2026-04-20   
 11090  ck_5b2dddfd766176af 2026-01-08 14:27:29      2026-04-

In [121]:
test.shape, test.nunique

((4909, 4),
 <bound method DataFrame.nunique of                 cookie_id   cookie_created_at window_start_ts window_end_ts
 0     ck_315fb710a0e371e7 2026-02-20 08:52:31      2026-04-20    2026-04-21
 1     ck_a76ee3b3e3e522fd 2026-01-19 13:42:15      2026-04-20    2026-04-21
 2     ck_94c9a4d382689e82 2026-04-19 06:28:37      2026-04-20    2026-04-21
 3     ck_8eaf9509ad9462a0 2026-01-19 17:18:06      2026-04-20    2026-04-21
 4     ck_9a88a5a989cb5bc6 2025-11-09 17:25:13      2026-04-20    2026-04-21
 ...                   ...                 ...             ...           ...
 4904  ck_8b772caed8f1541c 2026-03-06 06:17:32      2026-04-26    2026-04-27
 4905  ck_5403c76fe68a08ba 2025-09-16 01:17:03      2026-04-26    2026-04-27
 4906  ck_3e6833574edf1736 2026-02-06 19:45:02      2026-04-26    2026-04-27
 4907  ck_e527fbb9450a4581 2026-04-25 17:28:41      2026-04-26    2026-04-27
 4908  ck_cc90e219f1ebf390 2025-08-09 06:12:48      2026-04-26    2026-04-27
 
 [4909 rows x 4 columns]>)

 ## Перейдем к Feature Engineering

In [3]:
def mean_pairwise_distance(pts: np.ndarray) -> float: #для расчета расстояния между кликами
    n = len(pts)
    if n < 2:
        return np.nan
    d = np.sqrt(((pts[:, None, :] - pts[None, :, :]) ** 2).sum(-1))
    iu = np.triu_indices(n, k=1)
    return float(d[iu].mean())


def entropy_minute(ts: pd.Series) -> float:  #функции для агрегациия кликов
    ts = ts.dropna()
    if len(ts) < 2:
        return np.nan
    minutes = ts.dt.floor("min")
    counts = minutes.value_counts().to_numpy()
    p = counts / counts.sum()
    h = -(p * np.log(p)).sum()
    return float(h / np.log(len(counts))) if len(counts) > 1 else 0.0


def cv_inter_event(ts: pd.Series) -> float: #для анализа временных промежутков
    ts = ts.dropna().sort_values()
    if len(ts) < 3:
        return np.nan
    deltas = ts.diff().dropna().dt.total_seconds().to_numpy()
    if deltas.mean() == 0:
        return np.nan
    return float(deltas.std() / deltas.mean())


def mode_or_nan(s: pd.Series): #для категориальных признаков будет нужно вернуть моду
    s = s.dropna()
    return s.mode().iloc[0] if len(s) else np.nan


def parse_ua(ua: str) -> dict: #парсит столбец user-agent с помощью специальной библиотеки
    if not isinstance(ua, str) or not ua:
        return dict(browser=None, os=None, device=None,
                    is_mobile=False, is_pc=False, is_bot=False)
    u = parse(ua)
    return dict(
        browser=u.browser.family or None,
        os=u.os.family or None,
        device=u.device.family or None,
        is_mobile=u.is_mobile,
        is_pc=u.is_pc,
        is_bot=u.is_bot,
    )

CAT_COLS = ["platform", "event_name", "item_category", "item_location",
            "seller_type", "search_query"]
NUM_COLS = ["pointer_x", "pointer_y", "search_page", "eid", "item_id"]


def build_features(df_ev: pd.DataFrame, event_to_idx: dict) -> pd.DataFrame: #собирает все вместе
    df = df_ev.copy()

    # 1. Нормализация
    for c in CAT_COLS:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip().str.lower().replace({"nan": np.nan})
    for c in NUM_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df["event_ts"] = pd.to_datetime(df["event_ts"], errors="coerce")
    df = df.sort_values(["cookie_id", "event_ts"]).reset_index(drop=True)

    # 2. UA
    ua_parsed = df["user_agent"].apply(parse_ua).apply(pd.Series)
    df = pd.concat([df, ua_parsed], axis=1)

    # 3. Агрегация по cookie_id
    records = []
    for ck, g in df.groupby("cookie_id", sort=False):
        g = g.sort_values("event_ts")
        n_events = len(g)

        raw_ua = g["user_agent"].dropna()
        n_unique_ua = raw_ua.nunique()

        mask_click = g["pointer_x"].notna() & g["pointer_y"].notna()
        pts = g.loc[mask_click, ["pointer_x", "pointer_y"]].to_numpy()
        n_clicks = int(mask_click.sum())
        n_unique_points = g.loc[mask_click, ["pointer_x", "pointer_y"]].drop_duplicates().shape[0]

        ts = g["event_ts"].dropna()
        if len(ts) >= 2:
            span_min = (ts.max() - ts.min()).total_seconds() / 60
            n_events_per_min = n_events / span_min if span_min > 0 else np.nan
            std_inter_event = float(ts.sort_values().diff().dropna().dt.total_seconds().std())
        else:
            n_events_per_min = np.nan
            std_inter_event = np.nan

        cv = cv_inter_event(ts)
        n_unique_locations = g["item_location"].nunique(dropna=True)
        q = g["search_query"].dropna()

        target = g["target"].iloc[0] if "target" in g.columns else np.nan

        records.append(dict(
            cookie_id=ck,
            target=target,
            n_events=n_events,

            n_unique_ua=n_unique_ua,
            n_unique_browser=g["browser"].nunique(dropna=True),
            n_unique_os=g["os"].nunique(dropna=True),
            n_unique_device=g["device"].nunique(dropna=True),
            dominant_browser=mode_or_nan(g["browser"]),
            dominant_os=mode_or_nan(g["os"]),
            dominant_device=mode_or_nan(g["device"]),
            dominant_platform=mode_or_nan(g["platform"]),
            ua_changed=int(n_unique_ua > 1),
            is_mobile=bool(g["is_mobile"].any()),
            is_pc=bool(g["is_pc"].any()),
            is_bot=bool(g["is_bot"].any()),

            n_clicks=n_clicks,
            n_unique_points=n_unique_points,
            avg_distance=mean_pairwise_distance(pts),

            n_events_per_min=n_events_per_min,
            std_inter_event=std_inter_event,
            cv_inter_event=cv,
            regularity=1 - cv if pd.notna(cv) else np.nan,
            entropy_minute=entropy_minute(ts),

            n_visited_pages=g["search_page"].nunique(dropna=True),
            max_page=g["search_page"].max(skipna=True),

            n_unique_locations=n_unique_locations,
            avg_interactions_per_location=n_events / n_unique_locations if n_unique_locations else np.nan,
            n_unique_categories=g["item_category"].nunique(dropna=True),
            n_unique_items=g["item_id"].nunique(dropna=True),
            n_events_with_item_id=int(g["item_id"].notna().sum()),

            n_query_events=len(q),
            n_unique_query=q.nunique(),

            n_unique_event_names=g["event_name"].nunique(dropna=True),
            event_counts=g["event_name"].value_counts(dropna=True).to_dict(),
        ))

    agg = pd.DataFrame(records)

    def event_name_bins(d): #группировка событий по частотности
        out = {}
        for name, c in d.items():
            idx = event_to_idx.get(name)
            if idx is None:
                continue
            b = idx // 10
            out[b] = out.get(b, 0) + c
        return out

    if len(agg):
        bins_df = (agg["event_counts"].apply(event_name_bins)
                                     .apply(pd.Series).fillna(0).astype(int))
        bins_df.columns = [f"event_name_bin_{c}" for c in bins_df.columns]

        cnt = agg["event_counts"].apply(pd.Series).fillna(0).astype(int)
        cnt.columns = [f"cnt_{c}" for c in cnt.columns]

        agg = pd.concat([agg.drop(columns=["event_counts"]), bins_df, cnt], axis=1)
    else:
        agg = agg.drop(columns=["event_counts"], errors="ignore")

    return agg

## Сборка обучающего и тестового датасетов

In [5]:
event_order  = train_ev["event_name"].dropna().value_counts().index.tolist()
event_to_idx = {name: i for i, name in enumerate(event_order)}

agg      = build_features(train_ev, event_to_idx)
agg_test = build_features(test_ev,  event_to_idx)

print('agg:', agg.shape, '| agg_test:', agg_test.shape)

agg: (11091, 43) | agg_test: (4909, 43)


In [6]:
def align_to_meta(agg, meta, has_target=True):
    base = meta[["cookie_id"] + (["target"] if has_target else [])].copy()
    out = base.merge(agg, on="cookie_id", how="left", suffixes=("", "_agg"))

    if has_target:
        out["target"] = base["target"].values
    return out

agg_full      = align_to_meta(agg,      train, has_target=True)
agg_test_full = align_to_meta(agg_test, test,  has_target=False)

feat_cols = [c for c in agg_full.columns if c not in ("cookie_id", "target")]
feat_cols = [c for c in feat_cols if c in agg_test_full.columns]
agg_test_full = agg_test_full[["cookie_id"] + feat_cols]

print('после доклейки:')
print('  train:', agg_full.shape, '| NaN target:', agg_full.target.isna().sum())
print('  test :', agg_test_full.shape)
print('  признаков:', len(feat_cols))

missing = set(test.cookie_id) - set(agg_test_full.cookie_id)
assert not missing, f"Пропали cookie: {len(missing)}"

после доклейки:
  train: (11091, 44) | NaN target: 0
  test : (4909, 42)
  признаков: 41


## Подготовка данных для подачи в модель

In [18]:
X = agg_full[feat_cols].copy()
y = agg_full["target"].astype(int)

# для LightGBM готовим категориальные фичи, кодируем в int 
cat_features = []
for c in X.columns:
    if pd.api.types.is_bool_dtype(X[c]) or X[c].dtype == object:
        cat_features.append(c)
    elif pd.api.types.is_numeric_dtype(X[c]) and X[c].nunique(dropna=True) <= 2:
        cat_features.append(c)

# приводим категориальные к category dtype
for c in cat_features:
    all_vals = pd.concat([X[c], agg_test_full[c]], axis=0).astype("str")
    cats = pd.Categorical(all_vals).categories
    X[c]             = pd.Categorical(X[c].astype("str"),             categories=cats)
    agg_test_full[c] = pd.Categorical(agg_test_full[c].astype("str"), categories=cats)

print("cat_features:", cat_features)
print("X:", X.shape, "| positives:", y.mean().round(4))

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

cat_features: ['n_unique_os', 'n_unique_device', 'dominant_browser', 'dominant_os', 'dominant_device', 'dominant_platform', 'ua_changed', 'is_mobile', 'is_pc', 'is_bot']
X: (11091, 41) | positives: 0.0811


## Метрика

In [63]:
def precision_at_recall(y_true, y_score, recall_target=TARGET_RECALL):
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    mask = recall >= recall_target
    return float(precision[mask].max()) if mask.any() else np.nan


def evaluate(y_true, y_score): #дополнительные метрики
    return dict(
        P_at_R07=precision_at_recall(y_true, y_score),
        PR_AUC=average_precision_score(y_true, y_score),
        ROC_AUC=roc_auc_score(y_true, y_score),
    )

## Обучение первой модели - lgbm

In [76]:
def make_objective(X_tr, y_tr, X_va, y_va, cat_features, n_folds=1): #создаем objective для optuna

    from lightgbm import LGBMClassifier
    from sklearn.model_selection import StratifiedKFold

    if cat_features is not None:
        X_tr = X_tr.copy()
        X_va = X_va.copy()
        for c in cat_features:
            X_tr[c] = X_tr[c].astype("category")
            X_va[c] = X_va[c].astype("category")
#достаточно большой grid
    def objective(trial: optuna.Trial) -> float:
        params = {
            "metric":           "average_precision",  
            "boosting_type":    "gbdt",
            "verbosity":        -1,
            "random_state":     RANDOM_SEED,
            "n_jobs":           -1,

            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "num_leaves":       trial.suggest_int("num_leaves", 16, 256, log=True),
            "max_depth":        trial.suggest_int("max_depth", 3, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100, log=True),  
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),          
            "subsample":        trial.suggest_float("subsample", 0.5, 1.0),                 
            "subsample_freq":   trial.suggest_int("subsample_freq", 1, 10),                 
            "reg_alpha":        trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),    
            "reg_lambda":       trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),    
            "min_split_gain":   trial.suggest_float("min_split_gain", 0.0, 1.0),            
            "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 10.0, log=True),  
            "cat_smooth":       trial.suggest_int("cat_smooth", 1, 50),
            "cat_l2":           trial.suggest_float("cat_l2", 1e-3, 10.0, log=True),

            
            "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 15.0, log=True),

            
            "n_estimators":     5000,
            "early_stopping_rounds": 150,
        }

        if n_folds == 1:
            model = LGBMClassifier(**params)
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                eval_metric="average_precision",
                categorical_feature=cat_features if cat_features is not None else "auto",
            )
            p_va = model.predict_proba(
                X_va, num_iteration=model.best_iteration_
            )[:, 1]
            return precision_at_recall(y_va, p_va, TARGET_RECALL)

        #K-fold CV 
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_SEED)
        X_all = pd.concat([X_tr, X_va], axis=0).reset_index(drop=True)
        y_all = pd.concat([y_tr, y_va], axis=0).reset_index(drop=True)
        scores = []
        for tr_idx, va_idx in skf.split(X_all, y_all):
            X_fold_tr, X_fold_va = X_all.iloc[tr_idx], X_all.iloc[va_idx]
            y_fold_tr, y_fold_va = y_all.iloc[tr_idx], y_all.iloc[va_idx]

            
            if cat_features is not None:
                for c in cat_features:
                    X_fold_tr[c] = X_fold_tr[c].astype("category")
                    X_fold_va[c] = X_fold_va[c].astype("category")

            m = LGBMClassifier(**params)
            m.fit(
                X_fold_tr, y_fold_tr,
                eval_set=[(X_fold_va, y_fold_va)],
                eval_metric="average_precision",
                categorical_feature=cat_features if cat_features is not None else "auto",
            )
            p = m.predict_proba(
                X_fold_va, num_iteration=m.best_iteration_
            )[:, 1]
            scores.append(precision_at_recall(y_fold_va, p, TARGET_RECALL))
        return float(np.mean(scores))

    return objective

In [77]:
N_TRIALS = 150        
TIMEOUT  = 60 * 40    

sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED, multivariate=True)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=0)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    study_name="lgb_bots",
)

t0 = time.time()
study.optimize(
    make_objective(X_train, y_train, X_valid, y_valid, cat_features, n_folds=1),
    n_trials=N_TRIALS,
    timeout=TIMEOUT,
    show_progress_bar=True,
)
print(f"\nВремя: {(time.time() - t0)/60:.1f} мин")
print("Лучший P@R0.7 (holdout):", round(study.best_value, 4))
print("Лучшие параметры:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-09-28 21:45:27,443] A new study created in memory with name: lgb_bots


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-09-28 21:45:29,781] Trial 0 finished with value: 0.5906976744186047 and parameters: {'learning_rate': 0.02757359293934948, 'num_leaves': 223, 'max_depth': 10, 'min_child_samples': 29, 'colsample_bytree': 0.5780093202212182, 'subsample': 0.5779972601681014, 'subsample_freq': 1, 'reg_alpha': 2.9154431891537547, 'reg_lambda': 0.2537815508265665, 'min_split_gain': 0.7080725777960455, 'min_child_weight': 0.0012087541473056963, 'cat_smooth': 49, 'cat_l2': 2.1368329072358767, 'scale_pos_weight': 1.777174904859462}. Best is trial 0 with value: 0.5906976744186047.
[I 2026-09-28 21:45:33,248] Trial 1 finished with value: 0.5971563981042654 and parameters: {'learning_rate': 0.016362239850894633, 'num_leaves': 26, 'max_depth': 6, 'min_child_samples': 23, 'colsample_bytree': 0.7159725093210578, 'subsample': 0.645614570099021, 'subsample_freq': 7, 'reg_alpha': 0.003613894271216527, 'reg_lambda': 0.01474275315991467, 'min_split_gain': 0.3663618432936917, 'min_child_weight': 0.0667236717046420

In [78]:
study.best_params

{'learning_rate': 0.029487012859298358,
 'num_leaves': 20,
 'max_depth': 5,
 'min_child_samples': 26,
 'colsample_bytree': 0.9627591844933258,
 'subsample': 0.8908440808007067,
 'subsample_freq': 8,
 'reg_alpha': 0.007252260348793641,
 'reg_lambda': 0.0011176446148261627,
 'min_split_gain': 0.2822645804533064,
 'min_child_weight': 4.2624674525882,
 'cat_smooth': 22,
 'cat_l2': 0.006361923061487115,
 'scale_pos_weight': 1.5831565716349463}

In [79]:
from lightgbm import LGBMClassifier
best_params_lgb = {
    "metric":        "average_precision",
    "boosting_type": "gbdt",
    "verbosity":     -1,
    "random_state":  RANDOM_SEED,
    "n_jobs":        -1,
    "n_estimators":  5000,
    "early_stopping_rounds": 200,
    **study.best_params,
}

In [80]:
from lightgbm import LGBMClassifier

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_pred  = np.zeros(len(X))                # OOF вероятности ∈ [0,1]
test_pred = np.zeros(len(agg_test_full))    # усреднение по фолдам
fold_models, fold_iters = [], []


X_test = agg_test_full[feat_cols].copy()


_params = {k: v for k, v in best_params_lgb.items()
           if k not in ("n_estimators", "early_stopping_rounds")}
N_ESTIMATORS = best_params_lgb.get("n_estimators", 5000)
EARLY_STOP   = best_params_lgb.get("early_stopping_rounds", 200)


for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    X_tr = X_tr.copy()
    X_va = X_va.copy()
    X_te = X_test.copy()
    for c in cat_features:
        X_tr[c] = X_tr[c].astype("category")
        X_va[c] = X_va[c].astype("category")
        X_te[c] = X_te[c].astype("category")

    m_lgb = LGBMClassifier(
        **_params,
        n_estimators=N_ESTIMATORS,
        early_stopping_rounds=EARLY_STOP,
    )
    m_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="average_precision",
        categorical_feature=cat_features,
    )

    fold_models.append(m_lgb)
    fold_iters.append(m_lgb.best_iteration_)

    oof_pred[va_idx]  = m_lgb.predict_proba(
        X_va, num_iteration=m_lgb.best_iteration_
    )[:, 1]
    test_pred        += m_lgb.predict_proba(
        X_te, num_iteration=m_lgb.best_iteration_
    )[:, 1] / N_FOLDS

    print(f"fold {fold}: best_iter={m_lgb.best_iteration_} "
          f"P@R0.7={precision_at_recall(y_va, oof_pred[va_idx]):.4f} "
          f"PR-AUC={average_precision_score(y_va, oof_pred[va_idx]):.4f}")

print("\nOOF:",
      f"P@R0.7={precision_at_recall(y, oof_pred):.4f}",
      f"PR-AUC={average_precision_score(y, oof_pred):.4f}",
      f"ROC-AUC={roc_auc_score(y, oof_pred):.4f}")
print("fold iters:", fold_iters)
print("range oof_pred :", oof_pred.min().round(4), "—", oof_pred.max().round(4))
print("range test_pred:", test_pred.min().round(4), "—", test_pred.max().round(4))

#понадобится для blending
oof_lgb, test_lgb = oof_pred.copy(), test_pred.copy()

fold 1: best_iter=393 P@R0.7=0.7651 PR-AUC=0.7723
fold 2: best_iter=808 P@R0.7=0.7925 PR-AUC=0.7491
fold 3: best_iter=481 P@R0.7=0.7283 PR-AUC=0.7608
fold 4: best_iter=456 P@R0.7=0.5888 PR-AUC=0.7224
fold 5: best_iter=1024 P@R0.7=0.6087 PR-AUC=0.6958

OOF: P@R0.7=0.6767 PR-AUC=0.7364 ROC-AUC=0.9163
fold iters: [393, 808, 481, 456, 1024]
range oof_pred : 0.0 — 0.9997
range test_pred: 0.0003 — 0.9987


## Обучим XGBoost

In [81]:
def make_objective(X_tr, y_tr, X_va, y_va, cat_features, spw_baseline):
    dtrain = xgb.DMatrix(X_tr, y_tr, enable_categorical=True)
    dvalid = xgb.DMatrix(X_va, y_va, enable_categorical=True)
#тоже достаточно большой grid
    def objective(trial: optuna.Trial) -> float:
        params = {
            "objective":        "binary:logistic",
            "eval_metric":      "aucpr",          
            "tree_method":      "hist",          
            "verbosity":        0,
            "seed":             RANDOM_SEED,
            "nthread":          -1,

            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "max_depth":        trial.suggest_int("max_depth", 3, 12),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 50.0, log=True),
            "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "colsample_bylevel":trial.suggest_float("colsample_bylevel", 0.5, 1.0),
            "gamma":            trial.suggest_float("gamma", 1e-3, 10.0, log=True),
            "lambda":           trial.suggest_float("lambda", 1e-3, 10.0, log=True),   # L2
            "alpha":            trial.suggest_float("alpha", 1e-3, 10.0, log=True),    # L1
            "max_bin":          trial.suggest_int("max_bin", 128, 512, log=True),

            # балансировка классов — тюним вокруг baseline
            "scale_pos_weight": trial.suggest_float(
                "scale_pos_weight", 1.0, spw_baseline * 3, log=True
            ),

            # параметры категориальных
            "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 2, 20),
            "max_cat_threshold": trial.suggest_int("max_cat_threshold", 16, 128),
        }

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=5000,
            evals=[(dvalid, "valid")],
            early_stopping_rounds=150,
            verbose_eval=False,
        )
        p_va = model.predict(dvalid, iteration_range=(0, model.best_iteration + 1))
        return precision_at_recall(y_va, p_va, TARGET_RECALL)

    return objective

In [82]:
import xgboost as xgb
import time
import joblib
spw_default = (y_train == 0).sum() / (y_train == 1).sum()

In [83]:
N_TRIALS = 150 #ищем параметры
TIMEOUT  = 60 * 40

sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED, multivariate=True)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=10)

study_xgb = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    study_name="xgb_bots",
)

t0 = time.time()
study_xgb.optimize(
    make_objective(X_train, y_train, X_valid, y_valid, cat_features, spw_default),
    n_trials=N_TRIALS,
    timeout=TIMEOUT,
    show_progress_bar=True,
)
print(f"\nВремя: {(time.time()-t0)/60:.1f} мин")
print("Best P@R0.7:", round(study_xgb.best_value, 4))
for k, v in study_xgb.best_params.items():
    print(f"  {k}: {v}")

[I 2026-09-28 21:54:38,550] A new study created in memory with name: xgb_bots


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-09-28 21:54:40,142] Trial 0 finished with value: 0.5825688073394495 and parameters: {'learning_rate': 0.02757359293934948, 'max_depth': 12, 'min_child_weight': 17.524101118128137, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'colsample_bylevel': 0.5779972601681014, 'gamma': 0.0017073967431528124, 'lambda': 2.9154431891537547, 'alpha': 0.2537815508265665, 'max_bin': 341, 'scale_pos_weight': 1.0752995691184881, 'max_cat_to_onehot': 20, 'max_cat_threshold': 110}. Best is trial 0 with value: 0.5825688073394495.
[I 2026-09-28 21:54:41,832] Trial 1 finished with value: 0.654639175257732 and parameters: {'learning_rate': 0.01777174904859463, 'max_depth': 4, 'min_child_weight': 2.0492680115417348, 'subsample': 0.6521211214797689, 'colsample_bytree': 0.762378215816119, 'colsample_bylevel': 0.7159725093210578, 'gamma': 0.014618962793704957, 'lambda': 0.2801635158716261, 'alpha': 0.003613894271216527, 'max_bin': 191, 'scale_pos_weight': 3.64049155685678, 'max_c

In [84]:
best_params = {
    "objective":     "binary:logistic",
    "eval_metric":   "aucpr",
    "tree_method":   "hist",
    "verbosity":     0,
    "seed":          RANDOM_SEED,
    "nthread":       -1,
    **study_xgb.best_params,
}

In [87]:
N_FOLDS = 5 #считаем по фолдам
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

oof_pred  = np.zeros(len(X))
test_pred = np.zeros(len(agg_test_full))
fold_models = []
fold_iters  = []

X_test = agg_test_full[feat_cols].copy()


dtest = xgb.DMatrix(X_test, enable_categorical=True)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    dtr = xgb.DMatrix(X_tr, y_tr, enable_categorical=True)
    dva = xgb.DMatrix(X_va, y_va, enable_categorical=True)

    m_xgb = xgb.train(
        best_params,
        dtr,
        num_boost_round=5000,
        evals=[(dva, "valid")],
        early_stopping_rounds=200,
        verbose_eval=False,
    )
    fold_models.append(m_xgb)
    fold_iters.append(m_xgb.best_iteration)

    oof_pred[va_idx] = m_xgb.predict(dva, iteration_range=(0, m_xgb.best_iteration + 1))
    test_pred       += m_xgb.predict(dtest, iteration_range=(0, m_xgb.best_iteration + 1)) / N_FOLDS

    print(f"fold {fold}: best_iter={m.best_iteration} "
          f"P@R0.7={precision_at_recall(y_va, oof_pred[va_idx]):.4f} "
          f"PR-AUC={average_precision_score(y_va, oof_pred[va_idx]):.4f}")

print("\nOOF:",
      f"P@R0.7={precision_at_recall(y, oof_pred):.4f}",
      f"PR-AUC={average_precision_score(y, oof_pred):.4f}",
      f"ROC-AUC={roc_auc_score(y, oof_pred):.4f}")
print("fold iters:", fold_iters)

oof_xgb, test_xgb = oof_pred.copy(), test_pred.copy()

fold 1: best_iter=1024 P@R0.7=0.7651 PR-AUC=0.7653
fold 2: best_iter=1024 P@R0.7=0.7683 PR-AUC=0.7437
fold 3: best_iter=1024 P@R0.7=0.7159 PR-AUC=0.7398
fold 4: best_iter=1024 P@R0.7=0.5250 PR-AUC=0.7068
fold 5: best_iter=1024 P@R0.7=0.5727 PR-AUC=0.6839

OOF: P@R0.7=0.6625 PR-AUC=0.7247 ROC-AUC=0.9135
fold iters: [1709, 1039, 1258, 752, 1461]


## Взвешенное голосование моделей

In [90]:
from scipy.stats import rankdata #понадобится для ранжирования

In [95]:
def to_rank(x):
    return rankdata(x) / len(x)


oof_lgb_r = to_rank(oof_lgb)
oof_xgb_r = to_rank(oof_xgb)
test_lgb_r = to_rank(test_lgb)
test_xgb_r = to_rank(test_xgb)

In [97]:
oof_lgb_r

array([0.79325579, 0.63691281, 0.14840862, ..., 0.40555405, 0.60562618,
       0.3966279 ])

In [122]:
def blend_objective(trial):
    w = trial.suggest_float("w_lgb", 0.0, 1.0)
    oof_blend = w * oof_lgb_r + (1 - w) * oof_xgb_r
    return precision_at_recall(y, oof_blend, TARGET_RECALL)

study_blend = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    study_name="blend_lgb_xgb",
)
study_blend.optimize(blend_objective, n_trials=100, show_progress_bar=True)

w_lgb = study_blend.best_params["w_lgb"]
w_xgb = 1.0 - w_lgb
print(f"\nBest weights: LGB={w_lgb:.4f} | XGB={w_xgb:.4f}")
print(f"Best OOF P@R0.7 = {study_blend.best_value:.4f}")

oof_blend = w_lgb * oof_lgb_r + w_xgb * oof_xgb_r
test_blend = w_lgb * test_lgb_r + w_xgb * test_xgb_r

print("\nWeighted rank blend (optuna)")
print("BLEND OOF:", {k: round(v, 4) for k, v in evaluate(y, oof_blend).items()})

[I 2026-09-28 22:47:47,473] A new study created in memory with name: blend_lgb_xgb


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-09-28 22:47:47,487] Trial 0 finished with value: 0.670926517571885 and parameters: {'w_lgb': 0.3745401188473625}. Best is trial 0 with value: 0.670926517571885.
[I 2026-09-28 22:47:47,491] Trial 1 finished with value: 0.6803455723542117 and parameters: {'w_lgb': 0.9507143064099162}. Best is trial 1 with value: 0.6803455723542117.
[I 2026-09-28 22:47:47,497] Trial 2 finished with value: 0.6938325991189427 and parameters: {'w_lgb': 0.7319939418114051}. Best is trial 2 with value: 0.6938325991189427.
[I 2026-09-28 22:47:47,503] Trial 3 finished with value: 0.6847826086956522 and parameters: {'w_lgb': 0.5986584841970366}. Best is trial 2 with value: 0.6938325991189427.
[I 2026-09-28 22:47:47,508] Trial 4 finished with value: 0.6702127659574468 and parameters: {'w_lgb': 0.15601864044243652}. Best is trial 2 with value: 0.6938325991189427.
[I 2026-09-28 22:47:47,513] Trial 5 finished with value: 0.6702127659574468 and parameters: {'w_lgb': 0.15599452033620265}. Best is trial 2 with v

# Создание submission

In [101]:
assert (agg_test_full['cookie_id'].values == test['cookie_id'].values).all(), \
    "cookie_id order mismatch" #на всякий случай проверка

In [102]:
sub = pd.DataFrame({
    'cookie_id': agg_test_full['cookie_id'],
    'score': test_blend,
})

assert len(sub) == len(test), f"Length mismatch: {len(sub)} vs {len(test)}"
assert sub.score.between(0, 1).all(), "Scores outside [0, 1]"
assert sub.cookie_id.is_unique, "Duplicate cookie_ids"

sub.to_csv('submission.csv', index=False)
sub.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.044232
1,ck_a76ee3b3e3e522fd,0.729753
2,ck_94c9a4d382689e82,0.442550
3,ck_8eaf9509ad9462a0,0.040802
4,ck_9a88a5a989cb5bc6,0.573672
